# Task 2 Model Experimentation and Development - Global AI & Data Jobs Salary Dataset

## Objective

The purpose of this modelling task is to develop and evaluate a machine learning regression pipeline using PyCaret to predict the annual base salary of AI and data-related jobs.

The primary target variable is `salary_usd`. The modelling process will include preprocessing and transformation, model comparison using k-fold cross-validation, hyperparameter tuning, evaluation using multiple regression performance metrics, and selection of the best-performing model.

MLflow will also be used to track model experiments, metrics and artifacts throughout the model development process.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from hydra import compose, initialize

PROJECT_ROOT = Path.cwd().parents[1]

with initialize(version_base=None, config_path="../conf"):
    cfg = compose(config_name="config")

raw_data_path = PROJECT_ROOT / cfg.data.raw_path

df = pd.read_csv(raw_data_path)

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (90000, 35)


,id,country,job_role,ai_specialization,experience_level,experience_years,salary_usd,bonus_usd,education_required,industry,...,vacation_days,skill_demand_score,automation_risk,job_security_score,career_growth_score,work_life_balance_score,promotion_speed,salary_percentile,cost_of_living_index,employee_satisfaction
0,1,UAE,Machine Learning Engineer,Reinforcement Learning,Entry,0,66465,5395,Master,Automotive,...,27,12,76,57,65,73,15,55,1.23,76
1,2,USA,AI Engineer,LLM,Entry,1,75507,11713,Bootcamp,Retail,...,27,54,29,69,60,51,15,58,0.87,67
2,3,Brazil,Research Scientist,Analytics,Entry,0,41660,5268,PhD,Healthcare,...,13,12,49,70,59,68,37,13,2.13,61
3,4,India,Software Engineer AI,Computer Vision,Senior,6,43268,7975,Diploma,Tech,...,30,80,47,79,65,55,46,74,1.49,56
4,5,Germany,Machine Learning Engineer,Computer Vision,Entry,0,69119,4758,Master,Retail,...,24,82,47,64,52,69,17,21,0.87,72


## 1. Modelling Preparation

The target variable for this regression task is `salary_usd`.

Before modelling, `id` is excluded because it is only a unique identifier. Based on the EDA and feature definitions, `bonus_usd` and `salary_percentile` are also excluded as potential target-leakage features, since they may contain salary-related information that would not be independently available when predicting a new job record.

All remaining predictors are initially retained, including features with weak individual relationships with salary, as they may still contribute through non-linear relationships or interactions.

The EDA also identified several preprocessing considerations. `experience_level` has a clear ordinal structure, while the treatment of `education_required` and `company_size` will be defined during preprocessing. Since most numerical predictors do not show substantial skewness, numerical feature transformation will not be applied by default.

A separate experiment will evaluate a transformed `salary_usd` target because the target distribution showed moderate positive skewness.

### 1.1 Reserve Unseen Data

Before model experimentation, 10% of the original dataset is randomly reserved as unseen data using a fixed random state for reproducibility. This produces **81,000 records for model development** and **9,000 records for final unseen prediction**.

The reserved unseen data will not be used during preprocessing experimentation, model comparison, cross-validation, hyperparameter tuning, or model selection. This ensures that it remains independent from the model development process and can be used to assess the final selected pipeline on previously unseen records.

The remaining 90% of the dataset is used for model development. PyCaret will subsequently perform an 80/20 train-holdout split on this development dataset, while 10-fold cross-validation is performed using the training portion.

In [2]:
from sklearn.model_selection import train_test_split

model_data, unseen_data = train_test_split(
    df,
    test_size=0.10,
    random_state=cfg.model.random_state
)

target = cfg.model.target

# Features excluded due to identifier / potential target leakage
ignore_features = [
    "salary_percentile",
    "bonus_usd",
    "id"
]

print(f"Target variable: {target}")
print(f"Features excluded from modelling: {ignore_features}")
print(f"Original dataset shape: {df.shape}")
print("Model development data:", model_data.shape)
print("Reserved unseen data:", unseen_data.shape)

Target variable: salary_usd
Features excluded from modelling: ['salary_percentile', 'bonus_usd', 'id']
Original dataset shape: (90000, 35)
Model development data: (81000, 35)
Reserved unseen data: (9000, 35)


## 2. Experiment 1 - Baseline Models with All Non-Leakage Candidate Features

The first experiment establishes a baseline using all remaining candidate predictors after excluding `id` and the potential target-leakage features `bonus_usd` and `salary_percentile`.

Basic preprocessing required to represent the data appropriately is applied. Based on the feature definitions identified during EDA, `experience_level`, `education_required`, and `company_size` are treated as ordinal categorical features using explicitly defined category orderings. The remaining categorical variables are handled using PyCaret's standard categorical preprocessing.

The original `salary_usd` target is used without target transformation, automated feature selection, additional feature engineering, or optional numerical transformation. This provides a controlled baseline against which subsequent modelling experiments can be compared.

Later experiments will investigate whether techniques such as automated feature selection, feature engineering, alternative preprocessing strategies, and target transformation improve model performance.

In [4]:
import os
import mlflow

os.environ["GIT_PYTHON_REFRESH"] = "quiet" # does not complain about GIT login 
mlflow.set_tracking_uri("http://127.0.0.1:5000")


### 2.1 Initialise Baseline PyCaret Experiment

The baseline PyCaret regression experiment is initialised using an 80/20 train-holdout split with 10-fold cross-validation.

`id`, `bonus_usd`, and `salary_percentile` are excluded, while all remaining candidate predictors are retained. Based on the EDA, `experience_level`, `education_required`, and `company_size` are treated as ordinal features using predefined category orderings, while PyCaret handles the remaining categorical preprocessing automatically.

During 10-fold cross-validation, the training data is divided into 10 subsets. The model is trained on 9 folds and validated on the remaining fold, repeating this process until every fold has been used for validation. The average performance across the folds provides a more reliable estimate of model performance than relying on a single train-validation split.

No automated feature selection, feature engineering, numerical transformation, or target transformation is applied in this baseline experiment. A fixed random state ensures reproducibility, while MLflow is enabled to track experiment results.

In [5]:
from pycaret.regression import RegressionExperiment

ordinal_features = {
    "experience_level": [
        "Entry",
        "Mid",
        "Senior",
        "Lead"
    ],

    "education_required": [
        "Bootcamp",
        "Diploma",
        "Bachelor",
        "Master",
        "PhD"
    ],

    "company_size": [
        "Startup",
        "Small",
        "Medium",
        "Large",
        "Enterprise"
    ]
}

categorical_features = [
    "country",
    "job_role",
    "ai_specialization",
    "industry",
    "work_mode"
]

baseline_exp = RegressionExperiment()

baseline_exp.setup(
    data=model_data,
    target=target,
    ignore_features=ignore_features,
    ordinal_features=ordinal_features,
    categorical_features=categorical_features,
    train_size=1 - cfg.training.test_size,
    fold_strategy="kfold",
    fold=cfg.training.fold,
    fold_shuffle=True,
    session_id=cfg.model.random_state,
    log_experiment="mlflow",
    experiment_name="salary_baseline_non_leakage"
)

,Description,Value
0,Session id,42
1,Target,salary_usd
2,Target type,Regression
3,Original data shape,"(81000, 35)"
4,Transformed data shape,"(81000, 68)"
5,Transformed train set shape,"(64800, 68)"
6,Transformed test set shape,"(16200, 68)"
7,Ignore features,3
8,Ordinal features,3
9,Numeric features,23


2026/08/20 22:22:24 INFO mlflow.tracking.fluent: Experiment with name 'salary_baseline_non_leakage' does not exist. Creating a new experiment.


### 2.2 Baseline Model Comparison

Multiple regression algorithms are trained and evaluated using the same preprocessing pipeline and 10-fold cross-validation strategy defined in the baseline experiment.

The models are compared using MAE, MSE, RMSE, R², RMSLE, and MAPE. Among these metrics, `RMSE` is used as the primary metric for model comparison because it measures prediction error in the same USD scale as `salary_usd` while penalising larger errors more heavily than MAE. This is useful for salary prediction because the target contains legitimate high-salary observations, where large prediction errors would be particularly undesirable.

The remaining metrics are still considered to provide a broader assessment of model performance. MAE provides a more robust view of typical absolute error, while R² indicates the proportion of salary variation explained by the model.

The top-performing models are retained as candidates for subsequent modelling experiments and later hyperparameter tuning.

In [6]:
baseline_models = baseline_exp.compare_models(
    n_select=3,
    sort="RMSE"
)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
catboost,CatBoost Regressor,8880.5998,127908206.2225,11309.1373,0.9336,0.1092,0.0939,3.5490
gbr,Gradient Boosting Regressor,9218.4935,138638974.4099,11773.7290,0.9280,0.1184,0.0998,2.7200
rf,Random Forest Regressor,9205.0585,139850610.6207,11825.2763,0.9274,0.1130,0.0965,12.6450
et,Extra Trees Regressor,9388.3710,145472684.3690,12060.1925,0.9245,0.1159,0.0985,9.7710
lasso,Lasso Regression,11290.6555,207984150.8876,14421.4032,0.8920,0.2935,0.1433,1.3040
llar,Lasso Least Angle Regression,11290.6419,207984158.7269,14421.4035,0.8920,0.2935,0.1433,0.1800
br,Bayesian Ridge,11290.8968,207990344.0212,14421.6183,0.8920,0.2943,0.1433,0.4000
ridge,Ridge Regression,11291.2089,207990379.2421,14421.6195,0.8920,0.2937,0.1433,0.1680
lr,Linear Regression,11385.2900,211620929.3005,14540.9504,0.8902,0.3003,0.1447,0.7560
dt,Decision Tree Regressor,12342.3258,275360719.3922,16593.3799,0.8571,0.1580,0.1285,0.3710


2026/08/20 22:28:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/20 22:28:58 INFO mlflow.tracking._tracking_service.client: 🏃 View run CatBoost Regressor at: http://127.0.0.1:5000/#/experiments/675356573797874508/runs/01853ea89ed642859f3817899a308352.
2026/08/20 22:28:58 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/675356573797874508.
2026/08/20 22:29:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/20 22:29:21 INFO mlflow.tracking._tracking_service.client: 🏃 View run Gradient Boosting Regressor at: http://127.0.0.1:5000/#/experiments/675356573797874508/runs/b61c31bf8bef45bdb9fdb0f87e6501da.
2026/08/20 22:29:21 INFO mlflow.tracking._tracking_service.clien

### 2.3 Baseline Model Comparison Results

The baseline comparison shows that ensemble tree-based models perform best under 10-fold cross-validation. CatBoost achieved the lowest RMSE of approximately **$11,309.14**, the lowest MAE of **$8,880.60**, and the highest R² of **0.9336**, making it the strongest baseline model according to the primary ranking metric.

Gradient Boosting ranked second with an RMSE of approximately **$11,773.73** and R² of **0.9280**, while Random Forest ranked third with an RMSE of **$11,825.28** and R² of **0.9274**.

These ensemble models substantially outperform the regularised linear models, which achieved RMSE values of approximately **$14,421** and R² values of around **0.8920**. This suggests that non-linear relationships and interactions between predictors provide useful information for salary prediction.

CatBoost, Gradient Boosting, and Random Forest will therefore be retained as the strongest baseline candidates for comparison with the subsequent modelling experiments.

In [7]:
baseline_results = baseline_exp.pull()

baseline_results.style.format({
    "MAE": "{:,.2f}",
    "MSE": "{:,.2f}",
    "RMSE": "{:,.2f}",
    "R2": "{:.4f}",
    "RMSLE": "{:.4f}",
    "MAPE": "{:.4f}",
    "TT (Sec)": "{:.3f}"
})

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
catboost,CatBoost Regressor,"8,880.60","127,908,206.22","11,309.14",0.9336,0.1092,0.0939,3.549
gbr,Gradient Boosting Regressor,"9,218.49","138,638,974.41","11,773.73",0.9280,0.1184,0.0998,2.720
rf,Random Forest Regressor,"9,205.06","139,850,610.62","11,825.28",0.9274,0.1130,0.0965,12.645
et,Extra Trees Regressor,"9,388.37","145,472,684.37","12,060.19",0.9245,0.1159,0.0985,9.771
lasso,Lasso Regression,"11,290.66","207,984,150.89","14,421.40",0.8920,0.2935,0.1433,1.304
llar,Lasso Least Angle Regression,"11,290.64","207,984,158.73","14,421.40",0.8920,0.2935,0.1433,0.180
br,Bayesian Ridge,"11,290.90","207,990,344.02","14,421.62",0.8920,0.2943,0.1433,0.400
ridge,Ridge Regression,"11,291.21","207,990,379.24","14,421.62",0.8920,0.2937,0.1433,0.168
lr,Linear Regression,"11,385.29","211,620,929.30","14,540.95",0.8902,0.3003,0.1447,0.756
dt,Decision Tree Regressor,"12,342.33","275,360,719.39","16,593.38",0.8571,0.1580,0.1285,0.371


## 3. Experiment 2 - EDA-Guided Feature Selection and Feature Engineering

The second experiment evaluates whether reducing redundant or weakly informative predictors and combining closely related features can improve model performance.

Based on the EDA, `job_security_score` and `layoff_risk` are strongly related and represent similar aspects of employment stability. These two variables are replaced with a single engineered `employment_stability_score` to reduce redundancy while preserving their underlying information. In addition, `work_life_balance_score` (EDA 6.2) and `year` (EDA 7.3) are removed because the EDA suggested limited additional contribution relative to related predictors.

The remaining predictors are retained, including variables with weak individual correlations, as they may still contribute through non-linear relationships or interactions.

The same train-holdout split, 10-fold cross-validation strategy, preprocessing approach, and original `salary_usd` target are retained so that the effect of these feature refinements can be compared fairly against the baseline experiment.

The `employment_stability_score` combines `job_security_score` with the inverse of `layoff_risk` so that both components follow the same direction, where a higher value represents greater employment stability.

For example, if a job has a `job_security_score` of **80** and a `layoff_risk` of **0.20**, the layoff risk is first inverted and converted to a 0-100 scale:

`(1 - 0.20) × 100 = 80`

The two values are then averaged:

`(80 + 80) / 2 = 80`

The resulting `employment_stability_score` is **80**.

In [8]:
# Create Experiment 2 dataset
feature_data = model_data.copy()

# Engineer a combined employment stability feature
feature_data["employment_stability_score"] = (
    feature_data["job_security_score"]
    + (1 - feature_data["layoff_risk"]) * 100
) / 2

# Features removed based on EDA findings
manual_removed_features = [
    "job_security_score",
    "layoff_risk",
    "work_life_balance_score",
    "year"
]

feature_data.drop(
    columns=manual_removed_features,
    inplace=True
)

print("Baseline excluded features:", ignore_features)
print("Additional removed features:", manual_removed_features)
print("Engineered feature: employment_stability_score")
print("Experiment 2 dataset shape:", feature_data.shape)

Baseline excluded features: ['salary_percentile', 'bonus_usd', 'id']
Additional removed features: ['job_security_score', 'layoff_risk', 'work_life_balance_score', 'year']
Engineered feature: employment_stability_score
Experiment 2 dataset shape: (81000, 32)


### 3.1 Initialise Feature Selection PyCaret Experiment

In [9]:
feature_selection_exp = RegressionExperiment()

feature_selection_exp.setup(
    data=feature_data,
    target=target,
    ignore_features=ignore_features,
    ordinal_features=ordinal_features,
    categorical_features=categorical_features,
    train_size=1 - cfg.training.test_size,
    fold_strategy="kfold",
    fold=cfg.training.fold,
    fold_shuffle=True,
    session_id=cfg.model.random_state,
    log_experiment="mlflow",
    experiment_name="salary_feature_selection"
)

,Description,Value
0,Session id,42
1,Target,salary_usd
2,Target type,Regression
3,Original data shape,"(81000, 32)"
4,Transformed data shape,"(81000, 65)"
5,Transformed train set shape,"(64800, 65)"
6,Transformed test set shape,"(16200, 65)"
7,Ignore features,3
8,Ordinal features,3
9,Numeric features,20


2026/08/20 22:31:19 INFO mlflow.tracking.fluent: Experiment with name 'salary_feature_selection' does not exist. Creating a new experiment.


### 3.2 Feature Selection Model Comparison

In [10]:
feature_selection_models = feature_selection_exp.compare_models(
    n_select=3,
    sort="RMSE"
)

,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,22:31:23
Status,. . . . . . . . . . . . . . . . . .,Loading Dependencies
Estimator,. . . . . . . . . . . . . . . . . .,Compiling Library


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
catboost,CatBoost Regressor,8904.7089,128354142.2176,11328.8870,0.9334,0.1094,0.0941,3.7220
gbr,Gradient Boosting Regressor,9240.7659,139447580.9370,11807.9382,0.9276,0.1184,0.0999,2.6260
rf,Random Forest Regressor,9216.8252,140289909.4335,11843.7447,0.9272,0.1132,0.0966,13.4670
et,Extra Trees Regressor,9421.9575,146682969.5863,12110.3371,0.9239,0.1163,0.0988,11.1360
lr,Linear Regression,11344.9460,209663917.5393,14479.4552,0.8912,0.2922,0.1440,0.3260
llar,Lasso Least Angle Regression,11342.8829,209669993.1795,14479.6630,0.8912,0.2933,0.1440,0.1660
lasso,Lasso Regression,11342.9016,209670010.8938,14479.6636,0.8912,0.2933,0.1440,1.3410
ridge,Ridge Regression,11343.4252,209672641.3530,14479.7539,0.8912,0.2924,0.1440,0.1640
br,Bayesian Ridge,11343.1378,209672686.8226,14479.7554,0.8912,0.2929,0.1440,0.4290
dt,Decision Tree Regressor,12332.6233,274777661.1655,16575.6363,0.8574,0.1577,0.1284,0.3660


2026/08/20 22:38:17 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/20 22:38:17 INFO mlflow.tracking._tracking_service.client: 🏃 View run CatBoost Regressor at: http://127.0.0.1:5000/#/experiments/632020753518260600/runs/2eeae577d3a84b7bbc64a3381f523ba4.
2026/08/20 22:38:17 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/632020753518260600.
2026/08/20 22:38:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/20 22:38:39 INFO mlflow.tracking._tracking_service.client: 🏃 View run Gradient Boosting Regressor at: http://127.0.0.1:5000/#/experiments/632020753518260600/runs/ec2ad439163b4641b741fadd854dc01f.
2026/08/20 22:38:39 INFO mlflow.tracking._tracking_service.clien

### 3.3 Feature Selection and Engineering Model Comparison Results

The feature-refined experiment produced similar performance to the baseline, with CatBoost, Gradient Boosting, and Random Forest remaining the three strongest models. CatBoost achieved the lowest RMSE of approximately **$11,328.89**, followed by Gradient Boosting at **$11,807.94** and Random Forest at **$11,843.74**.

However, all three models performed slightly worse than their corresponding baseline results. CatBoost's RMSE increased from **$11,309.14 to $11,328.89**, Gradient Boosting from **$11,773.73 to $11,807.94**, and Random Forest from **$11,825.28 to $11,843.74**. Their MAE and R² values also showed small deteriorations.

This suggests that replacing `job_security_score` and `layoff_risk` with the engineered `employment_stability_score`, together with removing `work_life_balance_score` and `year`, resulted in a small loss of predictive information. Therefore, the original baseline feature configuration will be restored for **Experiment 3**, which evaluates whether normalisation and target transformation improve model performance.

In [11]:
feature_selection_results = feature_selection_exp.pull()

feature_selection_results.style.format({
    "MAE": "{:,.2f}",
    "MSE": "{:,.2f}",
    "RMSE": "{:,.2f}",
    "R2": "{:.4f}",
    "RMSLE": "{:.4f}",
    "MAPE": "{:.4f}",
    "TT (Sec)": "{:.3f}"
})

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
catboost,CatBoost Regressor,"8,904.71","128,354,142.22","11,328.89",0.9334,0.1094,0.0941,3.722
gbr,Gradient Boosting Regressor,"9,240.77","139,447,580.94","11,807.94",0.9276,0.1184,0.0999,2.626
rf,Random Forest Regressor,"9,216.83","140,289,909.43","11,843.74",0.9272,0.1132,0.0966,13.467
et,Extra Trees Regressor,"9,421.96","146,682,969.59","12,110.34",0.9239,0.1163,0.0988,11.136
lr,Linear Regression,"11,344.95","209,663,917.54","14,479.46",0.8912,0.2922,0.1440,0.326
llar,Lasso Least Angle Regression,"11,342.88","209,669,993.18","14,479.66",0.8912,0.2933,0.1440,0.166
lasso,Lasso Regression,"11,342.90","209,670,010.89","14,479.66",0.8912,0.2933,0.1440,1.341
ridge,Ridge Regression,"11,343.43","209,672,641.35","14,479.75",0.8912,0.2924,0.1440,0.164
br,Bayesian Ridge,"11,343.14","209,672,686.82","14,479.76",0.8912,0.2929,0.1440,0.429
dt,Decision Tree Regressor,"12,332.62","274,777,661.17","16,575.64",0.8574,0.1577,0.1284,0.366


## 4. Experiment 3 - Normalisation and Target Transformation

The third experiment evaluates whether additional preprocessing can improve model performance. Since Experiment 2 produced slightly weaker results, the original baseline feature configuration from Experiment 1 is restored.

Min-Max normalisation is applied to the numerical predictors to place them on a common scale. Although the EDA did not identify feature scaling as a specific requirement, normalisation is evaluated as an additional preprocessing step to determine whether scale-sensitive models benefit from having numerical variables represented on a consistent range.

In addition, a Yeo-Johnson transformation is applied to the `salary_usd` target because the EDA identified moderate positive skewness in its distribution. This transformation aims to reduce the skewness of the target and lessen the influence of very high salary values during model training.

The same train-holdout split, 10-fold cross-validation strategy, and random state are retained to ensure that the results can be compared fairly with the previous experiments.

### 4.1 Initialise Normalisation and Target Transformation PyCaret Experiment

In [12]:
preprocessing_exp = RegressionExperiment()

preprocessing_exp.setup(
    data=model_data,
    target=target,
    ignore_features=ignore_features,
    ordinal_features=ordinal_features,
    categorical_features=categorical_features,

    normalize=True,
    normalize_method="minmax",

    transform_target=True,

    train_size=1 - cfg.training.test_size,
    fold_strategy="kfold",
    fold=cfg.training.fold,
    fold_shuffle=True,
    session_id=cfg.model.random_state,

    log_experiment="mlflow",
    experiment_name="salary_normalization_target_transform"
)

,Description,Value
0,Session id,42
1,Target,salary_usd
2,Target type,Regression
3,Original data shape,"(81000, 35)"
4,Transformed data shape,"(81000, 68)"
5,Transformed train set shape,"(64800, 68)"
6,Transformed test set shape,"(16200, 68)"
7,Ignore features,3
8,Ordinal features,3
9,Numeric features,23


2026/08/20 23:52:40 INFO mlflow.tracking.fluent: Experiment with name 'salary_normalization_target_transform' does not exist. Creating a new experiment.


### 4.2 Normalisation and Target Transformation Model Comparison

In [13]:
preprocessing_models = preprocessing_exp.compare_models(
    n_select=3,
    sort="RMSE"
)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
catboost,CatBoost Regressor,8851.7861,126982793.7863,11268.0654,0.9341,0.1082,0.0929,3.8550
huber,Huber Regressor,9140.7710,135712233.2093,11649.0900,0.9296,0.1181,0.0985,1.5750
lr,Linear Regression,9159.7794,136124303.0471,11666.7192,0.9293,0.1179,0.0984,0.9450
br,Bayesian Ridge,9160.9611,136156230.8312,11668.0922,0.9293,0.1179,0.0984,0.4660
ridge,Ridge Regression,9161.1166,136166063.8248,11668.5130,0.9293,0.1179,0.0984,0.2260
gbr,Gradient Boosting Regressor,9082.3389,136187004.2207,11669.1567,0.9293,0.1119,0.0958,2.9830
rf,Random Forest Regressor,9211.1729,139897254.7578,11827.3026,0.9274,0.1128,0.0961,13.3710
et,Extra Trees Regressor,9399.6367,146023488.1105,12083.0536,0.9242,0.1158,0.0982,9.9730
ada,AdaBoost Regressor,12090.2270,255981077.1883,15998.0312,0.8671,0.1628,0.1336,3.1610
par,Passive Aggressive Regressor,12260.3908,267494989.3032,16132.2089,0.8612,0.1641,0.1301,0.2460


2026/08/20 23:59:34 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/20 23:59:34 INFO mlflow.tracking._tracking_service.client: 🏃 View run CatBoost Regressor at: http://127.0.0.1:5000/#/experiments/450861933026043037/runs/7502b7150b394c9c9f5f2a9318f9f480.
2026/08/20 23:59:34 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/450861933026043037.
2026/08/20 23:59:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/20 23:59:39 INFO mlflow.tracking._tracking_service.client: 🏃 View run Huber Regressor at: http://127.0.0.1:5000/#/experiments/450861933026043037/runs/ed91fa55edac4b348abc1ac58467579d.
2026/08/20 23:59:39 INFO mlflow.tracking._tracking_service.client: 🧪 View ex

### 4.2 Normalisation and Target Transformation Model Comparison Results

Under the normalisation and target-transformation configuration, CatBoost achieved the strongest performance with the lowest RMSE of approximately **$11,268.07**, the lowest MAE of **$8,851.79**, and the highest R² of **0.9341**.

Huber Regressor ranked second with an RMSE of approximately **$11,649.09** and R² of **0.9296**, while Linear Regression ranked third with an RMSE of approximately **$11,666.72** and R² of **0.9293**. 

The results also show that the additional preprocessing changes the relative performance of several algorithms, with Huber Regressor and Linear Regression performing considerably more competitively under this configuration.

In [14]:
preprocessing_results = preprocessing_exp.pull()

preprocessing_results.style.format({
    "MAE": "{:,.2f}",
    "MSE": "{:,.2f}",
    "RMSE": "{:,.2f}",
    "R2": "{:.4f}",
    "RMSLE": "{:.4f}",
    "MAPE": "{:.4f}",
    "TT (Sec)": "{:.3f}"
})

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
catboost,CatBoost Regressor,"8,851.79","126,982,793.79","11,268.07",0.9341,0.1082,0.0929,3.855
huber,Huber Regressor,"9,140.77","135,712,233.21","11,649.09",0.9296,0.1181,0.0985,1.575
lr,Linear Regression,"9,159.78","136,124,303.05","11,666.72",0.9293,0.1179,0.0984,0.945
br,Bayesian Ridge,"9,160.96","136,156,230.83","11,668.09",0.9293,0.1179,0.0984,0.466
ridge,Ridge Regression,"9,161.12","136,166,063.82","11,668.51",0.9293,0.1179,0.0984,0.226
gbr,Gradient Boosting Regressor,"9,082.34","136,187,004.22","11,669.16",0.9293,0.1119,0.0958,2.983
rf,Random Forest Regressor,"9,211.17","139,897,254.76","11,827.30",0.9274,0.1128,0.0961,13.371
et,Extra Trees Regressor,"9,399.64","146,023,488.11","12,083.05",0.9242,0.1158,0.0982,9.973
ada,AdaBoost Regressor,"12,090.23","255,981,077.19","15,998.03",0.8671,0.1628,0.1336,3.161
par,Passive Aggressive Regressor,"12,260.39","267,494,989.30","16,132.21",0.8612,0.1641,0.1301,0.246


## 5. Experiment Comparison and Selection of Final Modelling Setup

The top three models from each experiment are compared using the same evaluation metrics obtained from 10-fold cross-validation. RMSE remains the primary comparison metric, while MAE and R² are also considered when selecting the modelling configuration to carry forward for hyperparameter tuning.

In [16]:
# Select top 3 models from each experiment based on RMSE
baseline_top3 = (
    baseline_results
    .nsmallest(3, "RMSE")
    .assign(Experiment="Experiment 1 - Baseline")
)

feature_selection_top3 = (
    feature_selection_results
    .nsmallest(3, "RMSE")
    .assign(Experiment="Experiment 2 - Feature Refinement")
)

preprocessing_top3 = (
    preprocessing_results
    .nsmallest(3, "RMSE")
    .assign(Experiment="Experiment 3 - Preprocessing")
)

# Combine results
experiment_comparison = (
    pd.concat(
        [
            baseline_top3,
            feature_selection_top3,
            preprocessing_top3
        ],
        ignore_index=True
    )
    [["Experiment", "Model", "MAE", "RMSE", "R2", "RMSLE", "MAPE"]]
    .sort_values("RMSE", ascending=True)
    .reset_index(drop=True)
)

experiment_comparison.style.format({
    "MAE": "${:,.2f}",
    "RMSE": "${:,.2f}",
    "R2": "{:.4f}",
    "RMSLE": "{:.4f}",
    "MAPE": "{:.4f}"
})

,Experiment,Model,MAE,RMSE,R2,RMSLE,MAPE
0,Experiment 3 - Preprocessing,CatBoost Regressor,"$8,851.79","$11,268.07",0.9341,0.1082,0.0929
1,Experiment 1 - Baseline,CatBoost Regressor,"$8,880.60","$11,309.14",0.9336,0.1092,0.0939
2,Experiment 2 - Feature Refinement,CatBoost Regressor,"$8,904.71","$11,328.89",0.9334,0.1094,0.0941
3,Experiment 3 - Preprocessing,Huber Regressor,"$9,140.77","$11,649.09",0.9296,0.1181,0.0985
4,Experiment 3 - Preprocessing,Linear Regression,"$9,159.78","$11,666.72",0.9293,0.1179,0.0984
5,Experiment 1 - Baseline,Gradient Boosting Regressor,"$9,218.49","$11,773.73",0.9280,0.1184,0.0998
6,Experiment 2 - Feature Refinement,Gradient Boosting Regressor,"$9,240.77","$11,807.94",0.9276,0.1184,0.0999
7,Experiment 1 - Baseline,Random Forest Regressor,"$9,205.06","$11,825.28",0.9274,0.1130,0.0965
8,Experiment 2 - Feature Refinement,Random Forest Regressor,"$9,216.83","$11,843.74",0.9272,0.1132,0.0966


Across all experiments, CatBoost achieved the strongest performance. The best overall result was obtained in **Experiment 3 - Preprocessing**, where CatBoost achieved an RMSE of **$11,268.07**, MAE of **$8,851.79**, and R² of **0.9341**. This slightly outperformed CatBoost under the baseline configuration in Experiment 1, which achieved an RMSE of **$11,309.14**, and the feature-refined configuration in Experiment 2, which achieved an RMSE of **$11,328.89**.

Experiment 3 also improved the performance of several non-tree models. Huber Regressor and Linear Regression achieved RMSE values of **$11,649.09** and **$11,666.72** respectively, making them more competitive than under the previous configurations.

Based on the initial 10-fold cross-validation results, Experiment 3 currently provides the strongest modelling configuration, with CatBoost achieving the lowest RMSE of **$11,268.07**.

However, the CatBoost models from Experiments 1 and 2 also achieved competitive performance, so the three CatBoost configurations will be carried forward for hyperparameter tuning before selecting the final modelling pipeline.

## 6. Hyperparameter Tuning

Hyperparameter tuning is performed on the three strongest model configurations identified during the experiment comparison. All three use CatBoost but differ in their preprocessing pipelines:

- Experiment 1: Baseline preprocessing
- Experiment 2: Feature selection and engineering
- Experiment 3: Normalisation and target transformation

Each CatBoost model is tuned using the same cross-validation strategy and RMSE as the primary optimisation metric. The tuned results are then compared to determine the final model and preprocessing pipeline.

In [ ]:
baseline_catboost = baseline_exp.create_model(
    "catboost"
)

feature_catboost = feature_selection_exp.create_model(
    "catboost"
)

preprocessing_catboost = preprocessing_exp.create_model(
    "catboost"
)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,8885.7447,127616304.0177,11296.7386,0.9328,0.1093,0.0940
1,8804.5061,125397302.5985,11198.0937,0.9350,0.1092,0.0942
2,8967.8653,130737270.1062,11434.0400,0.9326,0.1091,0.0937
3,8829.3404,126809253.7224,11260.9615,0.9345,0.1094,0.0939
4,9024.0561,130420857.7680,11420.1952,0.9331,0.1105,0.0951
5,8793.3528,125723109.7657,11212.6317,0.9331,0.1094,0.0940
6,8745.0301,124869210.9266,11174.4893,0.9356,0.1081,0.0924
7,8917.0210,129353087.1319,11373.3499,0.9329,0.1084,0.0933
8,9040.8647,132152772.6776,11495.7719,0.9315,0.1103,0.0951


2026/08/21 00:10:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/21 00:10:47 INFO mlflow.tracking._tracking_service.client: 🏃 View run CatBoost Regressor at: http://127.0.0.1:5000/#/experiments/675356573797874508/runs/8160351bd1ec42928b6f352c7cfb088d.
2026/08/21 00:10:47 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/675356573797874508.


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,8907.0332,127819474.8801,11305.7275,0.9327,0.1094,0.0941
1,8804.3419,125883216.2700,11219.7690,0.9348,0.1091,0.0941
2,8966.5644,130365988.9079,11417.7926,0.9328,0.1092,0.0936
3,8862.2634,127541332.6516,11293.4199,0.9341,0.1096,0.0942
4,9042.0335,131023282.8063,11446.5402,0.9328,0.1105,0.0953
5,8803.7581,125332208.6094,11195.1869,0.9333,0.1095,0.0942
6,8782.9434,125380960.3200,11197.3640,0.9354,0.1084,0.0928
7,8964.8314,130280206.8434,11414.0355,0.9324,0.1087,0.0936
8,9061.3548,132044828.2665,11491.0760,0.9316,0.1105,0.0954


2026/08/21 00:11:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/21 00:11:35 INFO mlflow.tracking._tracking_service.client: 🏃 View run CatBoost Regressor at: http://127.0.0.1:5000/#/experiments/632020753518260600/runs/5fea0c22bc494988a9078460cb9ce46d.
2026/08/21 00:11:35 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/632020753518260600.


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,8865.1352,126872294.2058,11263.7602,0.9332,0.1085,0.0931
1,8744.8915,123523825.4777,11114.1273,0.9360,0.1077,0.0929
2,8942.9270,130212316.4424,11411.0611,0.9329,0.1083,0.0928
3,8798.4248,125461068.8986,11200.9405,0.9352,0.1083,0.0929
4,8983.6954,129239993.2328,11368.3769,0.9337,0.1094,0.0941
5,8747.0405,124445586.7701,11155.5182,0.9338,0.1081,0.0928
6,8717.0144,124005540.4844,11135.7775,0.9361,0.1070,0.0914
7,8917.7365,129572044.6358,11382.9717,0.9328,0.1076,0.0925
8,9008.1809,130880176.3372,11440.2874,0.9322,0.1093,0.0941


2026/08/21 00:12:17 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/21 00:12:17 INFO mlflow.tracking._tracking_service.client: 🏃 View run CatBoost Regressor at: http://127.0.0.1:5000/#/experiments/450861933026043037/runs/0363557ca99e4bd48501f94538ef4bd8.
2026/08/21 00:12:17 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/450861933026043037.


In [22]:
catboost_grid = {
    "iterations": [300, 500, 700],
    "depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "l2_leaf_reg": [3, 5, 7],
    "random_strength": [1, 2]
}

tuned_baseline_catboost = baseline_exp.tune_model(
    baseline_catboost,
    custom_grid=catboost_grid,
    optimize="RMSE",
    n_iter=20,
    fold=10,
    choose_better=True
)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,8854.7721,126126063.5838,11230.5861,0.9336,0.1086,0.0937
1,8719.2783,122714927.4519,11077.6770,0.9364,0.1080,0.0934
2,8931.4040,129445609.9385,11377.4167,0.9333,0.1085,0.0933
3,8802.8092,125745404.6944,11213.6258,0.9350,0.1089,0.0937
4,8970.2308,128629854.1888,11341.5102,0.9340,0.1096,0.0945
5,8757.9828,124147773.2406,11142.1620,0.9339,0.1086,0.0936
6,8697.6699,123258712.8408,11102.1941,0.9365,0.1072,0.0918
7,8919.8699,128662136.7142,11342.9333,0.9332,0.1079,0.0931
8,9010.1041,130351950.3531,11417.1779,0.9324,0.1098,0.0949


Fitting 10 folds for each of 20 candidates, totalling 200 fits


2026/08/21 01:07:04 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/21 01:07:04 INFO mlflow.tracking._tracking_service.client: 🏃 View run CatBoost Regressor at: http://127.0.0.1:5000/#/experiments/675356573797874508/runs/1a561f3f71334f528446bb15933dddc3.
2026/08/21 01:07:04 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/675356573797874508.


In [ ]:
print("Experiment 1 tuned parameters:")
print(tuned_baseline_catboost.get_params())

print("\nExperiment 2 tuned parameters:")
print(tuned_feature_catboost.get_params())

print("\nExperiment 3 tuned parameters:")
print(tuned_preprocessing_catboost.get_params())

tuned_baseline_results = baseline_exp.pull()
tuned_feature_results = feature_selection_exp.pull()
tuned_preprocessing_results = preprocessing_exp.pull()